In [1]:
import os
from dotenv import load_dotenv
from pprint import pprint

import pandas as pd

import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings

import google.generativeai as genai

from IPython.display import Markdown

In [2]:


load_dotenv()

api_key = os.getenv('GEMENI_API_KEY')
#print(api_key)
genai.configure(api_key=api_key)




In [3]:


for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
        print(m.name)



models/embedding-001
models/text-embedding-004
models/gemini-embedding-exp-03-07
models/gemini-embedding-exp
models/gemini-embedding-001


In [4]:
import json

with open(r'C:\Users\liali\YoloWdTagger\wdv3-timm\tag_desctiptions\eye_tags.json') as f:
    data= json.load(f)

print(data[0])

{'instruction': 'ringed eyes', 'input': '', 'output': 'Eyes in which the pupil and iris are composed of one or several rings or circles, usually concentric to the pupil. Eyes composed of many of these rings are frequently used to give a crazed appearance.'}


In [5]:


from_json = []

for item in data:
    entry = ""
    if item['instruction'] != '':
        entry += f"Instruction : {item['instruction']}\n"

    if item['input'] != '':
        entry += f"Input : {item['input']}\n"

    if item['output'] != '':
        entry += f"Output : {item['output']}"

    from_json.append(entry)

len(from_json)



12

In [6]:
print(from_json)

['Instruction : ringed eyes\nOutput : Eyes in which the pupil and iris are composed of one or several rings or circles, usually concentric to the pupil. Eyes composed of many of these rings are frequently used to give a crazed appearance.', 'Instruction : aqua eyes\nOutput : A character with blue-green colored eyes. Due to the subjective nature of color judgment and other factors (such as the lighting on the characters) there is an overlap between this tag, the blue eyes, and the green eyes tags.', 'Instruction : black eyes\nOutput : A character with black colored eyes. See bruised eye for eyes that are bruised, also known as a black eye. Also see solid circle pupils for eyes with irises that are completely flat black to the point of being indistinct from the pupil.', 'Instruction : blue eyes\nOutput : A character with blue colored eyes.Due to the subjective nature of color judgment and other factors (such as the lighting on the characters), there is an overlap between this tag and the

In [7]:
from PyPDF2 import PdfReader

def extract_text_from_pdf(file_path):
    pdf_reader = PdfReader(file_path)
    num_pages = len(pdf_reader.pages)

    page_offset = 0
    text = ""

    for page in range(page_offset, num_pages):
        text += pdf_reader.pages[page].extract_text()

    return text


text = extract_text_from_pdf(r'C:\Users\liali\YoloWdTagger\wdv3-timm\pdf_files\Tag Group_Eyes Tags Wiki _ Danbooru.pdf')
print(text)

Danbooru
My Account Posts Comments Notes Artists Tags Pools WikiForum
More »Search wiki pagesSearch New Changes Help |Posts (0) History Edit
Recent Changes ( all)
cheval grand (summer calm
navy drop) (umamusume)
saltire
spoon bending
til arrior
minoyama
rondoline e. effenberg (cosplay)
list of tales of... characters
tales of phantasia
rondoline e. effenberg
mimi baker
tales of legendia
tales of rebirth
saleh (tales)
sol 644
dracotail arthalion
dracotail lukias
aerial battle
spread arms
backbend
sapulaisimie
tenkyuu chimata pose
umamusume: cinderella gray
ceras yanagida lilienfeld
badboon
vanilla (vanillaklein)
Options
Wiki History
Discussions
What Links Heretag group:eyes tags
[See tag groups .]
Table of Contents
Iris
Individual colors of the iris
• aqua eyes
• black eyes
• blue eyes
• brown eyes
• green eyes
• grey eyes
• orange eyes
• purple eyes
• pink eyes
• red eyes
• white eyes
• yellow eyes
Multiple colors of the iris
• heterochromia
• multicolored eyes
• gradient eyes
• two-ton

In [8]:
def clean_extracted_text(text):
    cleaned_text = ""

    for i, line in enumerate(text.split('\n')):
        if len(line) > 10 and i > 70:
            cleaned_text += line + '\n'

    cleaned_text = cleaned_text.replace('.', '')
    cleaned_text = cleaned_text.replace('~', '')
    cleaned_text = cleaned_text.replace('©', '')
    cleaned_text = cleaned_text.replace('_', '')
    cleaned_text = cleaned_text.replace(';:;', '')
    return cleaned_text



In [9]:
cleaned_text = clean_extracted_text(text)
len(cleaned_text)

3229

In [10]:
pprint(cleaned_text)

('• orange pupils\n'
 '• pink pupils\n'
 '• purple pupils\n'
 '• red pupils\n'
 '• white pupils\n'
 '• yellow pupils\n'
 'Form of the pupils\n'
 '• constricted pupils\n'
 '• dilated pupils\n'
 '• extra pupils\n'
 '• horizontal pupils\n'
 '• no pupils\n'
 '• slit pupils\n'
 '• symbol-shaped pupils\n'
 '• diamond-shaped pupils\n'
 '• flower-shaped pupils\n'
 '• heart-shaped pupils\n'
 '• star-shaped pupils\n'
 '• solid circle pupils\n'
 '• cross-shaped pupils\n'
 '• x-shaped pupils\n'
 '• snowflakes-shapedpupils\n'
 '• power symbol-shaped pupils\n'
 '• crosshair pupils\n'
 '• mismatched pupils\n'
 '• blue sclera\n'
 '• black sclera\n'
 '• blank eyes  (white sclera)\n'
 '• bloodshot eyes\n'
 '• green sclera\n'
 '• mismatched sclera\n'
 '• no sclera\n'
 '• orange sclera\n'
 '• red sclera\n'
 '• yellow sclera\n'
 'Around the eyes\n'
 '• bags under eyes\n'
 '• aegyo sal\n'
 '• bruised eye\n'
 '• flaming eyes\n'
 '• glowing eyesTag Group:Eyes Tags Wiki | Danbooru '
 'https://danboorudonmaius/

In [11]:


from langchain.text_splitter import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
    add_start_index=True,
)



In [12]:


texts = text_splitter.create_documents([cleaned_text])
texts_from_json= text_splitter.create_documents(from_json)

In [16]:
documents = [chunk.page_content for chunk in texts] #Функция create_chroma_db(...) ожидает: documents: List[str]  # список строк

documents+= [chunk.page_content for chunk in texts_from_json]


print(type(texts))
print(type(texts_from_json))



<class 'list'>
<class 'list'>


Changes to the new embedding models

For the new embeddings model, embedding-001, there is a new task type parameter and the optional title (only valid with task_type=RETRIEVAL_DOCUMENT).

These new parameters apply only to the newest embeddings models.The task types are:
| Task Type           | Description                                                                 |
|---------------------|-----------------------------------------------------------------------------|
| RETRIEVAL_QUERY     | Specifies the given text is a query in a search/retrieval setting.          |
| RETRIEVAL_DOCUMENT  | Specifies the given text is a document in a search/retrieval setting.       |
| SEMANTIC_SIMILARITY | Specifies the given text will be used for Semantic Textual Similarity (STS).|
| CLASSIFICATION      | Specifies that the embeddings will be used for classification.              |
| CLUSTERING          | Specifies that the embeddings will be used for clustering.                  |

In [17]:
class GeminiEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        model = 'models/embedding-001'
        # for better results, try to provide a title for each input if the corpus is covering a lot of domains
        title = "tags"

        return genai.embed_content(
            model=model,
            content=input,
            task_type="retrieval_document",
            title=title)["embedding"]

In [18]:
import time
from tqdm import tqdm

def create_chroma_db(documents, name):
    chroma_client = chromadb.PersistentClient(path="../database/")

    db = chroma_client.get_or_create_collection(
        name=name, embedding_function=GeminiEmbeddingFunction())

    initiali_size = db.count()
    for i, d in tqdm(enumerate(documents), total=len(documents), desc="Creating Chroma DB"):
        db.add(
            documents=d,
            ids=str(i + initiali_size)
        )
        time.sleep(0.5)
    return db


def get_chroma_db(name):
    chroma_client = chromadb.PersistentClient(path="../database/")
    return chroma_client.get_collection(name=name, embedding_function=GeminiEmbeddingFunction())

In [19]:
db = create_chroma_db(documents, "first_try")

db.count()

C:\Users\liali\AppData\Local\Temp\ipykernel_15412\1825996937.py:8: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  name=name, embedding_function=GeminiEmbeddingFunction())
Creating Chroma DB: 100%|██████████| 16/16 [00:18<00:00,  1.18s/it]


16

In [20]:
# Вариант 1: Исключить embeddings
data = db.peek(3)
data_without_embeddings = {k: v for k, v in data.items() if k != 'embeddings'}
df = pd.DataFrame(data_without_embeddings)
print(df)

  ids                                          documents  uris    included  \
0   0  • orange pupils\n• pink pupils\n• purple pupil...  None   metadatas   
1   1  • button eyes\n• cephalopod eyes\n• compound e...  None   documents   
2   2  • covering own eyes\n• hair over eyes\n• hair ...  None  embeddings   

   data metadatas  
0  None      None  
1  None      None  
2  None      None  


In [21]:
def relevant_docs(query, db, n_results=2):
    passages = db.query(query_texts=[query], n_results=n_results)['documents'][0]
    return passages


In [64]:
question= "*_eyes"
passages= relevant_docs(question, db, 2)

Markdown(''.join(passages))


Instruction : green eyes
Output : A character with green colored eyes.Due to the subjective nature of color judgment and other factors (such as the lighting on the characters), there is an overlap between this tag and the aqua eyes tag.Instruction : pink eyes
Output : A character with pink colored eyes.Due to the subjective nature of color judgment and other factors (such as the lighting on the characters), there is an overlap between this tag and the purple eyes and red eyes tags.

In [53]:
def make_prompt(query, relevant_passage):
    escaped = relevant_passage.replace("'", "").replace('"', "")

    # === Variant 1: Minimal ===
    # Question only, no context
    # prompt = f"""Question: {query}.\n
    # Your answer:
    # """

    # === Variant 2: Standard with context and OUT OF CONTEXT option ===
    prompt = f"""
    You are a strict assistant. You are only allowed to answer questions if they are directly related to the context.

    Question: {query}

    Context:\n{escaped}

    IMPORTANT RULE:
    If the question is not directly related to the context above, you MUST respond only with: OUT OF CONTEXT

    Your answer:
    """

    # === Variant 3: Clarify if out of context, but still answer ===
    # prompt = f"""Question: {query}.\n
    # Additional information:\n {escaped}\n
    # If you find that the question has no relation to the additional information, you can ignore it and respond with 'OUT OF CONTEXT' if the question is unrelated in the first place, and then still answer the question, even if it's out of context, by clarifying to the user that the answer has no relation to the context.\n
    # Your answer:
    # """

    # === Variant 4: Environmental management system context ===
    # prompt = f"""The following questions are related to the environmental management system. Here is the question: {query}.\nTry to answer the question using the following additional information, which may help you formulate your answer.\nAdditional information:\n {escaped}
    # Your answer:
    # """

    return prompt


In [54]:
def convert_passage_to_str(passages:list):
    return ''.join([string+"\n" for string in passages])


In [55]:
promt= make_prompt(question, convert_passage_to_str(passages))
pprint(promt)

('\n'
 '    You are a strict assistant. You are only allowed to answer questions if '
 'they are directly related to the context.\n'
 '\n'
 '    Question: What is the moon?\n'
 '\n'
 '    Context:\n'
 '• covering own eyes\n'
 '• hair over eyes\n'
 '• hair over one eye\n'
 '• bandage over one eye\n'
 '• blindfold\n'
 '• hat over eyes\n'
 '• eyelashes\n'
 '• colored eyelashes\n'
 '• fake eyelashes\n'
 '• eyes visible through hair\n'
 '• eyeshadow\n'
 '• eye contact\n'
 '• looking afarTag Group:Eyes Tags Wiki | Danbooru '
 'https://danboorudonmaius/wikipages/taggroup%3Aeyestags\n'
 'Стр 4 из 6 30072025, 20:08• looking around\n'
 '• looking at another\n'
 '• looking at breasts\n'
 '• looking at hand\n'
 '• looking at hands\n'
 '• looking at mirror\n'
 '• looking at phone\n'
 '• looking at self\n'
 '• looking at viewer\n'
 '• looking at penis\n'
 '• looking at pussy\n'
 '• looking at crotch\n'
 '• looking afar\n'
 '• looking back\n'
 '• looking down\n'
 '• looking outside\n'
 '• looking ove

In [56]:
#pprint(list(genai.list_models()))

In [57]:


model= genai.GenerativeModel('models/gemini-2.0-flash-001')

answer = model.generate_content(promt)



In [58]:
Markdown(answer.text)

OUT OF CONTEXT


In [60]:


# Step 1
# question = "Donne-moi le nombre de planetes dans le systeme solaire"
question = "What is red eyes"

# Step 2
db = get_chroma_db("first_try")
passages = relevant_docs(question, db, n_results=5)

# Step 3
context = convert_passage_to_str(passages)

# Step 4
prompt = make_prompt(question, context)

# Step 5
model = genai.GenerativeModel('models/gemini-2.0-flash-001')
answer = model.generate_content(prompt)

# Step 6
Markdown(answer.text)



C:\Users\liali\AppData\Local\Temp\ipykernel_15412\1825996937.py:22: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  return chroma_client.get_collection(name=name, embedding_function=GeminiEmbeddingFunction())


A character with red colored eyes. Due to the subjective nature of color judgment and other factors (such as the lighting on the characters), there is an overlap between this tag and the orange eyes and brown eyes tags. See bloodshot eyes for eyes that are red from irritation.
